# r10 step E: locked test, once, only with the owner approval
About 1-2 h.

In [ ]:
import os, glob, json, hashlib, shutil, subprocess
W = '/kaggle/working'
PUB, PRIV = f'{W}/public', f'{W}/private'   # public: counts only, safe to commit; private: real question text, never commit
os.makedirs(PUB, exist_ok=True)
os.makedirs(PRIV, exist_ok=True)
REPO, BRANCH = '/kaggle/temp/Kinthread-llm-model', 'main'
R8P_SHA = '38c289d351d6517b5cde1bc9fb64ea97bd33eee193639d6351edda52fdfd7a52'


def sh(cmd):
    """Runs one shell command; stops the notebook if it fails."""
    print('$', cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f'STOP: command failed: {cmd}')


def stop(msg):
    raise SystemExit('STOP: ' + msg)


def par(*cmds):
    """Runs commands at once, one T4 each (one dev pass takes ~6 h; two in a row pass Kaggle's 12 h limit)."""
    procs = []
    for i, cmd in enumerate(cmds):
        print(f'$ [gpu {i % 2}]', cmd, flush=True)
        procs.append((cmd, subprocess.Popen(cmd, shell=True, env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(i % 2)})))
    bad = [cmd for cmd, p in procs if p.wait()]
    if bad:
        raise SystemExit(f'STOP: command failed: {bad[0]}')


def find(pattern):
    return sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))


def one(pattern, what):
    got = find(pattern)
    if not got:
        stop(f'attach {what} (no {pattern} under /kaggle/input)')
    return got[0]


def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(REPO):
    sh(f'git clone --depth 1 -b {BRANCH} https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}')
os.chdir(REPO)
tgz = find('real_questions_full.tgz')
if not tgz:
    stop('attach the private dataset kinthread-real-questions (it holds real_questions_full.tgz)')
sh(f'rm -rf data/real_questions && tar -xzf {tgz[0]}')
if not glob.glob('data/real_questions/*.jsonl'):
    stop('no real question files after unpacking real_questions_full.tgz')
print('repo and real questions ready')

ALLOW_LOCKED = False   # the owner sets this to True, once, after reading public/decision.json
dec = json.load(open(one('public/decision.json', 'the output of kinthread-r10-d-select')))
if not dec['adopt']:
    stop('decision.json says keep r8p: there is nothing to score (r8p was already scored once)')
if not ALLOW_LOCKED:
    stop('the locked test is scored once and only with the owner approval: set ALLOW_LOCKED = True')
R9P = one('runs/v10-r10/best.pt', 'the output of kinthread-r10-c-train')
if sha(R9P) != dec['r10_sha256']:
    stop('the attached r10 checkpoint is not the one step D selected')
VER = one('public/r10_verifier.json', 'the output of kinthread-r10-d-select')
FLAGS = f"--adaptive 1.0 --hybrid --verifier {VER} --cover --split_search --para 2 --k 16 {dec['extra_flags']}"

In [ ]:
sh(f'python -m eval.run_real --ckpt {R9P} --split test {FLAGS} --out {PRIV}/r10_test.json')
test = json.load(open(f'{PRIV}/r10_test.json'))['score']['splits']
summary = {'step': 'E', 'r10_test': test, 'r8p_test': {'test fair': '474/689 = 68.8%', 'test verified': '462/640 = 72.2%'},
           'setting': dec['config'], 'extra_flags': dec['extra_flags'], 'r10_sha256': dec['r10_sha256']}
json.dump(summary, open(f'{PUB}/E_summary.json', 'w'), indent=1)
print(json.dumps(summary, indent=1))
print('E DONE')